# Positional subspaces pilot (single-concept, WordNet construction)

A concept has one subspace for each word-token position. Generation compares these
subspaces with overlapping hidden-state windows, without word-boundary detection.
This is a positional product-subspace adaptation, not tensor unfolding or GDS.
See [Gatto et al.](https://arxiv.org/abs/1909.01954) for the related product-Grassmann framework.

Concepts come from a JSON file — either the validated concept pool
`resources/32_concept_pool_bf16.json` (`pool[].synset`) or a `concepts/*.json`
word-cloud file (`[].synset`). Only the synset is used; concepts are built with the
existing WordNet construction under the **model's own language set** (Llama → 8 WordNet
languages, matching notebook 32), so the pool concepts populate every token position.
The `words` clouds are reserved for a later words-based construction. Each concept is
evaluated on its own — no blending and no differential (repel) guides. Arms:
positional (raw and hidden-state-normalized score), a reversed-position control, and
pooled subspaces at rank `R` and rank `T·R`.

The primary score averages **projection magnitudes**; the ablation divides each
magnitude by the hidden-state norm. Neither score is squared energy. Shared
pooled-reference cosine and continuation-only PPL are the comparison metrics.

**Limitation:** after summing all overlapping windows, interior states have been
compared with every active positional subspace. Permuting subspaces changes
individual windows but, for the fixed-denominator score, cumulative differences
come from the left boundary. The zero-padded right tail includes all available
contributions. The reversed-position control makes this limitation measurable.

Execute manually with CUDA and cached model weights. This notebook does not download
missing model weights. `SMOKE=True` selects one prompt and one concept.
Each execution creates a new artifact directory; historical results are preserved.

In [ ]:
SMOKE = True
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
QUANTIZATION = "bf16"
# Validated concept pool from notebook 32 (Llama 8-language WordNet, MIN_LEMMAS=11).
# Also accepts a concepts/*.json word-cloud file (list of {synset, words}).
CONCEPT_FILE = "resources/32_concept_pool_bf16.json"
MAX_TOKENS = 3
RANK = 8
MIN_LEMMAS = 11
K = 4
STEPS = 24
SEED = 0
LAMBDAS = [0.0, 0.4]

In [ ]:
import hashlib
import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.experiments.comparison_metrics import evaluate_continuation, ppl_metrics
from frames.experiments.notebook_helpers import llama_chat
from frames.experiments.steering_comparison import clean_continuation
from frames.representations.frame import FrameUnembeddingRepresentation
from subspace import PositionalConceptSubspace, SubspaceUnembeddingRepresentation

torch.manual_seed(SEED)

## Synthetic order diagnostic (CPU only)


In [ ]:
synthetic = PositionalConceptSubspace(
    synset="synthetic", bases=torch.eye(3).unsqueeze(-1), sample_counts=[1, 1, 1]
)
reverse_synthetic = PositionalConceptSubspace(
    synset="synthetic-reversed", bases=synthetic.bases.flip(0), sample_counts=[1, 1, 1]
)
h = torch.randn(12, 3)
h[:2] = 0
assert not torch.allclose(synthetic.project(h), reverse_synthetic.project(h))
assert torch.allclose(
    synthetic.project(h).sum(), reverse_synthetic.project(h).sum(), atol=1e-6
)
h[0] = torch.tensor([1.0, 0.0, 0.0])
assert not torch.allclose(
    synthetic.project(h).sum(), reverse_synthetic.project(h).sum()
)
pd.DataFrame(
    {"original": synthetic.project(h), "reversed": reverse_synthetic.project(h)}
)

## One shared model and identical eligible word sets


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "The model pilot requires CUDA; the preceding diagnostic is CPU-only."
    )
if QUANTIZATION != "bf16":
    raise ValueError(
        "This pilot is specified for bf16; use a separate experiment for AWQ."
    )
# No language override: from_model_id picks the model's set (Llama -> 8 languages),
# matching the concept pool's construction regime in notebook 32.
fur = FrameUnembeddingRepresentation.from_model_id(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization=None,
)
fur.model.model.eval()
sur = SubspaceUnembeddingRepresentation(
    model=fur.model, unembedding=fur, data=fur.data, rank=RANK
)
sur_wide = SubspaceUnembeddingRepresentation(
    model=fur.model, unembedding=fur, data=fur.data, rank=MAX_TOKENS * RANK
)
all_synsets = fur.data.get_all_synsets(fur.model.tokenizer, MIN_LEMMAS, MAX_TOKENS)

# Concepts come from the JSON file (synset field only; words / pool metadata ignored
# under the WordNet construction). Accepts either the notebook-32 pool ({"pool": [...]})
# or a concepts/*.json word-cloud list. Synsets not eligible at this
# MIN_LEMMAS / MAX_TOKENS are skipped rather than erroring.
requested = json.loads((ROOT / CONCEPT_FILE).read_text())
entries = requested["pool"] if isinstance(requested, dict) else requested
wanted = [e["synset"] for e in entries]
eligible = set(all_synsets["synset"])
names = [s for s in wanted if s in eligible]
skipped = [s for s in wanted if s not in eligible]
if not names:
    raise ValueError(f"No requested concept is eligible at MIN_LEMMAS={MIN_LEMMAS}")

pooled, wide, positional, reversed_spaces = {}, {}, {}, {}
construction = {}
for name in names:
    subset = all_synsets[all_synsets["synset"] == name]
    pooled[name] = sur.compute_concept(subset)
    wide[name] = sur_wide.compute_concept(subset)
    positional[name] = sur.compute_concept(
        subset, method="positional", max_token_count=MAX_TOKENS
    )
    c = positional[name]
    reversed_spaces[name] = PositionalConceptSubspace(
        synset=name,
        bases=c.bases.flip(0),
        sample_counts=c.sample_counts[::-1],
        metadata={"control": "reversed_positions", "source": c.construction_metadata()},
    )
    construction[name] = {
        **c.construction_metadata(),
        "eligible_token_rows": [rows.tolist() for rows in subset["tokens"]],
        "pooled_effective_rank": int((pooled[name].basis.norm(dim=0) > 0).sum()),
        "wide_effective_rank": int((wide[name].basis.norm(dim=0) > 0).sum()),
    }
print(f"Built {len(names)} concepts; skipped (not eligible): {skipped}")
pd.DataFrame(
    [
        {
            "concept": n,
            "position": j + 1,
            "samples": c.sample_counts[j],
            "effective_rank": c.effective_ranks[j],
        }
        for n, c in positional.items()
        for j in range(MAX_TOKENS)
    ]
)

In [ ]:
prompt_texts = ["Tell me a story about", "My favorite thing is"]
prompts = [{"text": llama_chat(p), "prefix": "", "user": p} for p in prompt_texts]
cases = [{"name": n, "positive": [n], "negative": []} for n in names]
if SMOKE:
    prompts, cases = prompts[:1], cases[:1]
settings = {
    "model": MODEL_ID,
    "quantization": QUANTIZATION,
    "language": fur.data.language_codes.name,
    "concept_file": CONCEPT_FILE,
    "concepts": [case["name"] for case in cases],
    "skipped_ineligible": skipped,
    "max_tokens": MAX_TOKENS,
    "rank_cap": RANK,
    "min_lemmas": MIN_LEMMAS,
    "k": K,
    "steps": STEPS,
    "seed": SEED,
    "lambdas": LAMBDAS,
    "smoke": SMOKE,
    "survivor_rule": "greedy_one_step_lookahead",
    "rtol": 1e-3,
    "positional_score": "active_position_mean_projection_magnitude",
    "normalization_ablation": [False, True],
    "tail": "right_zero_pad_fixed_denominator",
    "metric": f"shared_pooled_rank{RANK}_projection_cosine_and_continuation_ppl",
    "torch_version": str(torch.__version__),
    "model_revision": getattr(fur.model.model.config, "_commit_hash", None),
}
source_paths = [
    "subspace/positional.py",
    "subspace/concept.py",
    "subspace/representation.py",
    "subspace/generation.py",
    CONCEPT_FILE,
    "subspace/notebooks/positional_subspace_pilot.ipynb",
]
manifest = {
    "settings": settings,
    "construction": construction,
    "prompts": prompts,
    "cases": cases,
    "source_sha256": {
        p: hashlib.sha256((ROOT / p).read_bytes()).hexdigest() for p in source_paths
    },
}
identity = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[
    :12
]
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + identity
out = ROOT / "resources" / "positional_subspace_pilot_unquantized" / run_name
out.mkdir(parents=True, exist_ok=False)
(out / "manifest.json").write_text(json.dumps(manifest, indent=2))
harness = EvaluationHarness(fur, out / "unused_harness.jsonl", MIN_LEMMAS, MAX_TOKENS)
references = {f"shared_pooled_rank{RANK}": {n: c.basis for n, c in pooled.items()}}

## Generation and shared evaluation

Each concept is steered on its own. Arms: positional (raw and hidden-state-normalized
score), a reversed-position control, and pooled subspaces at rank `R` and rank `T·R`.
Rank `T·R` is a storage-budget control, not a guarantee of equal effective rank or
equal representational capacity. Fluency weight (lambda) has the same selection role
across arms, but raw score scales differ between the raw and normalized variants.

In [ ]:
def clean(text, prompt):
    return clean_continuation(text, prompt, fur.model.tokenizer.all_special_tokens)


def ppl(text):
    ids = fur.model.tokenizer(text, add_special_tokens=False)["input_ids"]
    if not text.strip() or len(ids) < 2:
        return None
    value = harness.perplexity(text)
    return value if math.isfinite(value) else None


baselines = [
    clean(fur.generate([p["text"]], max_new_tokens=STEPS + 1, do_sample=False)[0], p)
    for p in prompts
]
(out / "baselines.json").write_text(json.dumps(baselines, indent=2))
rows = []


def record(text, prompt_index, case, method, lam, normalize, probe):
    baseline = baselines[prompt_index]
    metrics = evaluate_continuation(
        fur, text, baseline, references, case["positive"], case["negative"]
    )
    row = {
        "run_id": identity,
        "method": method,
        "case": case["name"],
        "prompt_index": prompt_index,
        "lambda": lam,
        "normalize_score": normalize,
        "continuation": text,
        "concept_trajectory": probe,
        "cosine": metrics,
        "ppl": ppl_metrics(ppl(text), ppl(baseline)),
    }
    with (out / "generations.jsonl").open("a") as stream:
        stream.write(json.dumps(row, allow_nan=False) + "\n")
    rows.append(row)


with torch.inference_mode():
    for case in cases:
        for i, baseline in enumerate(baselines):
            record(baseline, i, case, "unsteered", None, None, None)
        for method, concepts in [
            ("positional", positional),
            ("positional_reversed", reversed_spaces),
            (f"pooled_rank{RANK}", pooled),
            (f"pooled_rank{MAX_TOKENS * RANK}", wide),
        ]:
            guide = concepts[case["name"]]
            for normalize in [False, True]:
                for lam in LAMBDAS:
                    for i, prompt in enumerate(prompts):
                        texts, probe = sur.generate_with_topk_guide(
                            [prompt["text"]],
                            guide,
                            k=K,
                            steps=STEPS,
                            fluency_weight=lam,
                            normalize_score=normalize,
                        )
                        record(
                            clean(texts[0], prompt),
                            i,
                            case,
                            method,
                            lam,
                            normalize,
                            probe[0].tolist(),
                        )

In [ ]:
summary = pd.json_normalize(rows)
summary.drop(columns=["concept_trajectory"]).to_csv(out / "summary.csv", index=False)
summary[["method", "case", "lambda", "normalize_score", "continuation"]]